In [1]:
{
  "cells": [
    {
      "cell_type": "code",
      "execution_count": 1,
      "metadata": {
        "colab": {
          "base_uri": "https://localhost:8080/"
        },
        "id": "2qJm5sdEtu6z",
        "outputId": "8d37cebc-cc98-40e9-8af2-642739771e14"
      },
      "outputs": [
        {
          "output_type": "stream",
          "name": "stdout",
          "text": [
            "\u001b[2K     \u001b[90m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[32m88.8/88.8 kB\u001b[0m \u001b[31m1.9 MB/s\u001b[0m eta \u001b[36m0:00:00\u001b[0m\n",
            "\u001b[2K   \u001b[90m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[32m395.5/395.5 kB\u001b[0m \u001b[31m9.4 MB/s\u001b[0m eta \u001b[36m0:00:00\u001b[0m\n",
            "\u001b[2K   \u001b[90m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[32m341.3/341.3 kB\u001b[0m \u001b[31m9.5 MB/s\u001b[0m eta \u001b[36m0:00:00\u001b[0m\n",
            "\u001b[?25h"
          ]
        }
      ],
      "source": [
        "!pip install -q langchain langchain-text-splitters tiktoken pypdf fpdf2"
      ],
      "id": "2qJm5sdEtu6z"
    },
    {
      "cell_type": "code",
      "execution_count": 2,
      "metadata": {
        "colab": {
          "base_uri": "https://localhost:8080/"
        },
        "id": "CAfbB3bYtu67",
        "outputId": "138afab7-b79c-4683-c423-628b9360b018"
      },
      "outputs": [
        {
          "output_type": "stream",
          "name": "stdout",
          "text": [
            "Document created: docs/sample_article.txt\n",
            "Total characters: 1786\n",
            "Total words: 262\n"
          ]
        }
      ],
      "source": [
        "import os\n",
        "\n",
        "os.makedirs(\"docs\", exist_ok=True)\n",
        "\n",
        "article_text = \"\"\"Artificial Intelligence: An Overview\n",
        "\n",
        "Artificial Intelligence, or AI, is the field of computer science focused on building systems\n",
        "that can perform tasks which normally require human intelligence. These tasks include\n",
        "understanding language, recognizing images, making decisions, and learning from experience.\n",
        "\n",
        "Machine Learning is a subset of AI where systems learn patterns from data instead of following\n",
        "hardcoded rules. Instead of programming every decision by hand, we feed the system many examples,\n",
        "and it learns the underlying pattern on its own. This approach powers everything from spam filters\n",
        "to recommendation systems.\n",
        "\n",
        "Deep Learning takes Machine Learning further by using neural networks with many layers. These\n",
        "layered networks can automatically discover complex patterns in large amounts of data, such as\n",
        "recognizing faces in photos or understanding the meaning of a sentence.\n",
        "\n",
        "Natural Language Processing, or NLP, is the branch of AI that focuses specifically on human\n",
        "language. NLP powers chatbots, translation tools, and search engines. A key building block of\n",
        "modern NLP is the embedding, which converts words or sentences into numeric vectors that capture\n",
        "meaning.\n",
        "\n",
        "Retrieval-Augmented Generation, or RAG, is a technique that combines a search system with a\n",
        "language model. Instead of relying only on what a language model memorized during training, RAG\n",
        "first retrieves relevant, up-to-date information from a document collection, then passes that\n",
        "information to the model so it can generate a more accurate, grounded answer.\n",
        "\n",
        "Vector Databases store embeddings and allow fast similarity search across millions of documents.\n",
        "They are a core infrastructure piece behind modern RAG systems, chatbots, and recommendation\n",
        "engines used by companies around the world today.\n",
        "\"\"\"\n",
        "\n",
        "with open(\"docs/sample_article.txt\", \"w\") as f:\n",
        "    f.write(article_text)\n",
        "\n",
        "print(\"Document created: docs/sample_article.txt\")\n",
        "print(\"Total characters:\", len(article_text))\n",
        "print(\"Total words:\", len(article_text.split()))"
      ],
      "id": "CAfbB3bYtu67"
    },
    {
      "cell_type": "code",
      "execution_count": 3,
      "metadata": {
        "colab": {
          "base_uri": "https://localhost:8080/"
        },
        "id": "pnpq8Fj4tu7A",
        "outputId": "0d11f615-5e33-4a28-cd33-64140fb013a4"
      },
      "outputs": [
        {
          "output_type": "stream",
          "name": "stdout",
          "text": [
            "Loaded document preview:\n",
            "\n",
            "Artificial Intelligence: An Overview\n",
            "\n",
            "Artificial Intelligence, or AI, is the field of computer science focused on building systems\n",
            "that can perform tasks which normally require human intelligence. These tasks include\n",
            "understanding language, recognizing images, making decisions, and learning from exp ...\n"
          ]
        }
      ],
      "source": [
        "with open(\"docs/sample_article.txt\", \"r\") as f:\n",
        "    loaded_text = f.read()\n",
        "\n",
        "print(\"Loaded document preview:\\n\")\n",
        "print(loaded_text[:300], \"...\")"
      ],
      "id": "pnpq8Fj4tu7A"
    },
    {
      "cell_type": "code",
      "execution_count": 4,
      "metadata": {
        "colab": {
          "base_uri": "https://localhost:8080/"
        },
        "id": "PUoQa42qtu7E",
        "outputId": "4eca7949-3724-494b-83bf-c0f2ac686fbf"
      },
      "outputs": [
        {
          "output_type": "stream",
          "name": "stdout",
          "text": [
            "PDF created: docs/sample_article.pdf\n"
          ]
        }
      ],
      "source": [
        "from fpdf import FPDF\n",
        "\n",
        "pdf = FPDF()\n",
        "pdf.add_page()\n",
        "pdf.set_font(\"Times\", size=12)\n",
        "\n",
        "for line in article_text.split(\"\\n\"):\n",
        "    # Using an explicit width (e.g., 190mm for A4 page with margins) instead of 0\n",
        "    pdf.multi_cell(190, 8, line)\n",
        "\n",
        "pdf.output(\"docs/sample_article.pdf\")\n",
        "print(\"PDF created: docs/sample_article.pdf\")"
      ],
      "id": "PUoQa42qtu7E"
    },
    {
      "cell_type": "code",
      "execution_count": 5,
      "metadata": {
        "colab": {
          "base_uri": "https://localhost:8080/"
        },
        "id": "c8n3RUuQtu7F",
        "outputId": "6513d378-6c95-4b21-b058-0943eb42abe7"
      },
      "outputs": [
        {
          "output_type": "stream",
          "name": "stdout",
          "text": [
            "Pages found: 1\n",
            "\n",
            "Extracted PDF text preview:\n",
            "\n",
            "Artificial Intelligence: An Overview\n",
            "Artificial Intelligence, or AI, is the field of computer science focused on building systems\n",
            "that can perform tasks which normally require human intelligence. These tasks include\n",
            "understanding language, recognizing images, making decisions, and learning from expe ...\n"
          ]
        }
      ],
      "source": [
        "from pypdf import PdfReader\n",
        "\n",
        "reader = PdfReader(\"docs/sample_article.pdf\")\n",
        "\n",
        "pdf_text = \"\"\n",
        "for page in reader.pages:\n",
        "    pdf_text += page.extract_text() + \"\\n\"\n",
        "\n",
        "print(\"Pages found:\", len(reader.pages))\n",
        "print(\"\\nExtracted PDF text preview:\\n\")\n",
        "print(pdf_text[:300], \"...\")"
      ],
      "id": "c8n3RUuQtu7F"
    },
    {
      "cell_type": "code",
      "execution_count": 6,
      "metadata": {
        "colab": {
          "base_uri": "https://localhost:8080/"
        },
        "id": "Vkq-d8IRtu7H",
        "outputId": "fb3c692e-24aa-4905-c6cb-582493e09bf3"
      },
      "outputs": [
        {
          "output_type": "stream",
          "name": "stdout",
          "text": [
            "Number of chunks: 8\n",
            "\n",
            "--- Chunk 1 ---\n",
            " Artificial Intelligence: An Overview\n",
            "\n",
            "Artificial Intelligence, or AI, is the field of computer science focused on building systems\n",
            "that can perform tasks which normally require human intelligence. These tasks include\n",
            "understanding language, recognizing images, making decisions, and learning from exp\n",
            "\n",
            "--- Chunk 2 ---\n",
            " ng images, making decisions, and learning from experience.\n",
            "\n",
            "Machine Learning is a subset of AI where systems learn patterns from data instead of following\n",
            "hardcoded rules. Instead of programming every decision by hand, we feed the system many examples,\n",
            "and it learns the underlying pattern on its own\n"
          ]
        }
      ],
      "source": [
        "def fixed_size_chunking(text, chunk_size=300, overlap=50):\n",
        "    chunks = []\n",
        "    start = 0\n",
        "    while start < len(text):\n",
        "        end = start + chunk_size\n",
        "        chunks.append(text[start:end])\n",
        "        start += chunk_size - overlap\n",
        "    return chunks\n",
        "\n",
        "fixed_chunks = fixed_size_chunking(loaded_text, chunk_size=300, overlap=50)\n",
        "\n",
        "print(\"Number of chunks:\", len(fixed_chunks))\n",
        "print(\"\\n--- Chunk 1 ---\\n\", fixed_chunks[0])\n",
        "print(\"\\n--- Chunk 2 ---\\n\", fixed_chunks[1])"
      ],
      "id": "Vkq-d8IRtu7H"
    },
    {
      "cell_type": "code",
      "execution_count": 7,
      "metadata": {
        "colab": {
          "base_uri": "https://localhost:8080/"
        },
        "id": "YseU3AJ1tu7J",
        "outputId": "3040dd86-715a-4225-f957-0191fcd89755"
      },
      "outputs": [
        {
          "output_type": "stream",
          "name": "stderr",
          "text": [
            "WARNING:langchain_text_splitters.base:Created a chunk of size 318, which is longer than the specified 300\n",
            "WARNING:langchain_text_splitters.base:Created a chunk of size 359, which is longer than the specified 300\n"
          ]
        },
        {
          "output_type": "stream",
          "name": "stdout",
          "text": [
            "Number of chunks: 7\n",
            "\n",
            "--- Chunk 1 (36 chars) ---\n",
            "Artificial Intelligence: An Overview\n",
            "\n",
            "--- Chunk 2 (270 chars) ---\n",
            "Artificial Intelligence, or AI, is the field of computer science focused on building systems\n",
            "that can perform tasks which normally require human intelligence. These tasks include\n",
            "understanding language, recognizing images, making decisions, and learning from experience.\n",
            "\n",
            "--- Chunk 3 (318 chars) ---\n",
            "Machine Learning is a subset of AI where systems learn patterns from data instead of following\n",
            "hardcoded rules. Instead of programming every decision by hand, we feed the system many examples,\n",
            "and it learns the underlying pattern on its own. This approach powers everything from spam filters\n",
            "to recommendation systems.\n",
            "\n",
            "--- Chunk 4 (260 chars) ---\n",
            "Deep Learning takes Machine Learning further by using neural networks with many layers. These\n",
            "layered networks can automatically discover complex patterns in large amounts of data, such as\n",
            "recognizing faces in photos or understanding the meaning of a sentence.\n",
            "\n",
            "--- Chunk 5 (291 chars) ---\n",
            "Natural Language Processing, or NLP, is the branch of AI that focuses specifically on human\n",
            "language. NLP powers chatbots, translation tools, and search engines. A key building block of\n",
            "modern NLP is the embedding, which converts words or sentences into numeric vectors that capture\n",
            "meaning.\n",
            "\n",
            "--- Chunk 6 (359 chars) ---\n",
            "Retrieval-Augmented Generation, or RAG, is a technique that combines a search system with a\n",
            "language model. Instead of relying only on what a language model memorized during training, RAG\n",
            "first retrieves relevant, up-to-date information from a document collection, then passes that\n",
            "information to the model so it can generate a more accurate, grounded answer.\n",
            "\n",
            "--- Chunk 7 (239 chars) ---\n",
            "Vector Databases store embeddings and allow fast similarity search across millions of documents.\n",
            "They are a core infrastructure piece behind modern RAG systems, chatbots, and recommendation\n",
            "engines used by companies around the world today.\n"
          ]
        }
      ],
      "source": [
        "from langchain_text_splitters import CharacterTextSplitter\n",
        "\n",
        "char_splitter = CharacterTextSplitter(\n",
        "    separator=\"\\n\\n\",\n",
        "    chunk_size=300,\n",
        "    chunk_overlap=50\n",
        ")\n",
        "\n",
        "char_chunks = char_splitter.split_text(loaded_text)\n",
        "\n",
        "print(\"Number of chunks:\", len(char_chunks))\n",
        "for i, chunk in enumerate(char_chunks):\n",
        "    print(f\"\\n--- Chunk {i+1} ({len(chunk)} chars) ---\")\n",
        "    print(chunk)"
      ],
      "id": "YseU3AJ1tu7J"
    },
    {
      "cell_type": "code",
      "execution_count": 8,
      "metadata": {
        "colab": {
          "base_uri": "https://localhost:8080/"
        },
        "id": "xOtfc7_8tu7K",
        "outputId": "a60c388d-2195-4f0b-9e0e-5f0153c60592"
      },
      "outputs": [
        {
          "output_type": "stream",
          "name": "stdout",
          "text": [
            "Number of chunks: 9\n",
            "\n",
            "--- Chunk 1 (36 chars) ---\n",
            "Artificial Intelligence: An Overview\n",
            "\n",
            "--- Chunk 2 (270 chars) ---\n",
            "Artificial Intelligence, or AI, is the field of computer science focused on building systems\n",
            "that can perform tasks which normally require human intelligence. These tasks include\n",
            "understanding language, recognizing images, making decisions, and learning from experience.\n",
            "\n",
            "--- Chunk 3 (291 chars) ---\n",
            "Machine Learning is a subset of AI where systems learn patterns from data instead of following\n",
            "hardcoded rules. Instead of programming every decision by hand, we feed the system many examples,\n",
            "and it learns the underlying pattern on its own. This approach powers everything from spam filters\n",
            "\n",
            "--- Chunk 4 (26 chars) ---\n",
            "to recommendation systems.\n",
            "\n",
            "--- Chunk 5 (260 chars) ---\n",
            "Deep Learning takes Machine Learning further by using neural networks with many layers. These\n",
            "layered networks can automatically discover complex patterns in large amounts of data, such as\n",
            "recognizing faces in photos or understanding the meaning of a sentence.\n",
            "\n",
            "--- Chunk 6 (291 chars) ---\n",
            "Natural Language Processing, or NLP, is the branch of AI that focuses specifically on human\n",
            "language. NLP powers chatbots, translation tools, and search engines. A key building block of\n",
            "modern NLP is the embedding, which converts words or sentences into numeric vectors that capture\n",
            "meaning.\n",
            "\n",
            "--- Chunk 7 (281 chars) ---\n",
            "Retrieval-Augmented Generation, or RAG, is a technique that combines a search system with a\n",
            "language model. Instead of relying only on what a language model memorized during training, RAG\n",
            "first retrieves relevant, up-to-date information from a document collection, then passes that\n",
            "\n",
            "--- Chunk 8 (77 chars) ---\n",
            "information to the model so it can generate a more accurate, grounded answer.\n",
            "\n",
            "--- Chunk 9 (239 chars) ---\n",
            "Vector Databases store embeddings and allow fast similarity search across millions of documents.\n",
            "They are a core infrastructure piece behind modern RAG systems, chatbots, and recommendation\n",
            "engines used by companies around the world today.\n"
          ]
        }
      ],
      "source": [
        "from langchain_text_splitters import RecursiveCharacterTextSplitter\n",
        "\n",
        "recursive_splitter = RecursiveCharacterTextSplitter(\n",
        "    chunk_size=300,\n",
        "    chunk_overlap=50,\n",
        "    separators=[\"\\n\\n\", \"\\n\", \". \", \" \", \"\"]\n",
        ")\n",
        "\n",
        "recursive_chunks = recursive_splitter.split_text(loaded_text)\n",
        "\n",
        "print(\"Number of chunks:\", len(recursive_chunks))\n",
        "for i, chunk in enumerate(recursive_chunks):\n",
        "    print(f\"\\n--- Chunk {i+1} ({len(chunk)} chars) ---\")\n",
        "    print(chunk)"
      ],
      "id": "xOtfc7_8tu7K"
    },
    {
      "cell_type": "code",
      "execution_count": 9,
      "metadata": {
        "colab": {
          "base_uri": "https://localhost:8080/"
        },
        "id": "5vl0r6IUtu7N",
        "outputId": "330d2e80-0ac5-4d94-c0ab-5421394c672d"
      },
      "outputs": [
        {
          "output_type": "stream",
          "name": "stdout",
          "text": [
            "Number of chunks: 7\n",
            "\n",
            "--- Chunk 1 ---\n",
            "Artificial Intelligence: An Overview\n",
            "\n",
            "Artificial Intelligence, or AI, is the field of computer science focused on building systems\n",
            "that can perform tasks which normally require human intelligence. These tasks include\n",
            "understanding language, recognizing images, making decisions, and learning from experience.\n",
            "\n",
            "--- Chunk 2 ---\n",
            "Machine Learning is a subset of AI where systems learn patterns from data instead of following\n",
            "hardcoded rules. Instead of programming every decision by hand, we feed the system many examples,\n",
            "and it learns the underlying pattern on its own.\n",
            "\n",
            "--- Chunk 3 ---\n",
            "This approach powers everything from spam filters\n",
            "to recommendation systems. Deep Learning takes Machine Learning further by using neural networks with many layers.\n",
            "\n",
            "--- Chunk 4 ---\n",
            "These\n",
            "layered networks can automatically discover complex patterns in large amounts of data, such as\n",
            "recognizing faces in photos or understanding the meaning of a sentence. Natural Language Processing, or NLP, is the branch of AI that focuses specifically on human\n",
            "language.\n"
          ]
        }
      ],
      "source": [
        "import re\n",
        "\n",
        "def sentence_chunking(text, sentences_per_chunk=2):\n",
        "    # Simple sentence splitter using punctuation (no extra downloads needed)\n",
        "    sentences = re.split(r'(?<=[.!?])\\s+', text.strip())\n",
        "    sentences = [s for s in sentences if s]\n",
        "\n",
        "    chunks = []\n",
        "    for i in range(0, len(sentences), sentences_per_chunk):\n",
        "        chunk = \" \".join(sentences[i:i + sentences_per_chunk])\n",
        "        chunks.append(chunk)\n",
        "    return chunks\n",
        "\n",
        "sentence_chunks = sentence_chunking(loaded_text, sentences_per_chunk=2)\n",
        "\n",
        "print(\"Number of chunks:\", len(sentence_chunks))\n",
        "for i, chunk in enumerate(sentence_chunks[:4]):\n",
        "    print(f\"\\n--- Chunk {i+1} ---\")\n",
        "    print(chunk)"
      ],
      "id": "5vl0r6IUtu7N"
    },
    {
      "cell_type": "code",
      "execution_count": 10,
      "metadata": {
        "colab": {
          "base_uri": "https://localhost:8080/"
        },
        "id": "GzkGzcxntu7P",
        "outputId": "081eba13-3665-45fb-f78b-f693aa48544e"
      },
      "outputs": [
        {
          "output_type": "stream",
          "name": "stdout",
          "text": [
            "Number of chunks: 7\n",
            "\n",
            "--- Chunk 1 (60 tokens) ---\n",
            "Artificial Intelligence: An Overview\n",
            "\n",
            "Artificial Intelligence, or AI, is the field of computer science focused on building systems\n",
            "that can perform tasks which normally require human intelligence. These tasks include\n",
            "understanding language, recognizing images, making decisions, and learning from experience.\n",
            "\n",
            "Machine Learning is a subset of\n",
            "\n",
            "--- Chunk 2 (60 tokens) ---\n",
            " learning from experience.\n",
            "\n",
            "Machine Learning is a subset of AI where systems learn patterns from data instead of following\n",
            "hardcoded rules. Instead of programming every decision by hand, we feed the system many examples,\n",
            "and it learns the underlying pattern on its own. This approach powers everything from spam filters\n",
            "to recommendation\n",
            "\n",
            "--- Chunk 3 (60 tokens) ---\n",
            " This approach powers everything from spam filters\n",
            "to recommendation systems.\n",
            "\n",
            "Deep Learning takes Machine Learning further by using neural networks with many layers. These\n",
            "layered networks can automatically discover complex patterns in large amounts of data, such as\n",
            "recognizing faces in photos or understanding the meaning of a sentence.\n",
            "\n",
            "Natural Language\n"
          ]
        }
      ],
      "source": [
        "import tiktoken\n",
        "\n",
        "encoding = tiktoken.get_encoding(\"cl100k_base\")\n",
        "\n",
        "def token_chunking(text, max_tokens=60, overlap_tokens=10):\n",
        "    tokens = encoding.encode(text)\n",
        "    chunks = []\n",
        "    start = 0\n",
        "    while start < len(tokens):\n",
        "        end = start + max_tokens\n",
        "        chunk_tokens = tokens[start:end]\n",
        "        chunks.append(encoding.decode(chunk_tokens))\n",
        "        start += max_tokens - overlap_tokens\n",
        "    return chunks\n",
        "\n",
        "token_chunks = token_chunking(loaded_text, max_tokens=60, overlap_tokens=10)\n",
        "\n",
        "print(\"Number of chunks:\", len(token_chunks))\n",
        "for i, chunk in enumerate(token_chunks[:3]):\n",
        "    token_count = len(encoding.encode(chunk))\n",
        "    print(f\"\\n--- Chunk {i+1} ({token_count} tokens) ---\")\n",
        "    print(chunk)"
      ],
      "id": "GzkGzcxntu7P"
    },
    {
      "cell_type": "code",
      "execution_count": 11,
      "metadata": {
        "colab": {
          "base_uri": "https://localhost:8080/",
          "height": 226
        },
        "id": "oJ1ZT7r-tu7R",
        "outputId": "f7197e5a-42bd-4f82-9765-1888390f9286"
      },
      "outputs": [
        {
          "output_type": "execute_result",
          "data": {
            "text/plain": [
              "                         Strategy  Number of Chunks  Avg Chunk Length (chars)\n",
              "0            Fixed-size (scratch)                 8                     265.2\n",
              "1           CharacterTextSplitter                 7                     253.3\n",
              "2  RecursiveCharacterTextSplitter                 9                     196.8\n",
              "3                  Sentence-based                 7                     253.4\n",
              "4                     Token-based                 7                     305.0"
            ],
            "text/html": [
              "\n",
              "  <div id=\"df-ea456e67-3306-47c0-92c4-3a41ff950044\" class=\"colab-df-container\">\n",
              "    <div>\n",
              "<style scoped>\n",
              "    .dataframe tbody tr th:only-of-type {\n",
              "        vertical-align: middle;\n",
              "    }\n",
              "\n",
              "    .dataframe tbody tr th {\n",
              "        vertical-align: top;\n",
              "    }\n",
              "\n",
              "    .dataframe thead th {\n",
              "        text-align: right;\n",
              "    }\n",
              "</style>\n",
              "<table border=\"1\" class=\"dataframe\">\n",
              "  <thead>\n",
              "    <tr style=\"text-align: right;\">\n",
              "      <th></th>\n",
              "      <th>Strategy</th>\n",
              "      <th>Number of Chunks</th>\n",
              "      <th>Avg Chunk Length (chars)</th>\n",
              "    </tr>\n",
              "  </thead>\n",
              "  <tbody>\n",
              "    <tr>\n",
              "      <th>0</th>\n",
              "      <td>Fixed-size (scratch)</td>\n",
              "      <td>8</td>\n",
              "      <td>265.2</td>\n",
              "    </tr>\n",
              "    <tr>\n",
              "      <th>1</th>\n",
              "      <td>CharacterTextSplitter</td>\n",
              "      <td>7</td>\n",
              "      <td>253.3</td>\n",
              "    </tr>\n",
              "    <tr>\n",
              "      <th>2</th>\n",
              "      <td>RecursiveCharacterTextSplitter</td>\n",
              "      <td>9</td>\n",
              "      <td>196.8</td>\n",
              "    </tr>\n",
              "    <tr>\n",
              "      <th>3</th>\n",
              "      <td>Sentence-based</td>\n",
              "      <td>7</td>\n",
              "      <td>253.4</td>\n",
              "    </tr>\n",
              "    <tr>\n",
              "      <th>4</th>\n",
              "      <td>Token-based</td>\n",
              "      <td>7</td>\n",
              "      <td>305.0</td>\n",
              "    </tr>\n",
              "  </tbody>\n",
              "</table>\n",
              "</div>\n",
              "    <div class=\"colab-df-buttons\">\n",
              "\n",
              "  <div class=\"colab-df-container\">\n",
              "    <button class=\"colab-df-convert\" onclick=\"convertToInteractive('df-ea456e67-3306-47c0-92c4-3a41ff950044')\"\n",
              "            title=\"Convert this dataframe to an interactive table.\"\n",
              "            style=\"display:none;\">\n",
              "\n",
              "  <svg xmlns=\"http://www.w3.org/2000/svg\" height=\"24px\" viewBox=\"0 -960 960 960\">\n",
              "    <path d=\"M120-120v-720h720v720H120Zm60-500h600v-160H180v160Zm220 220h160v-160H400v160Zm0 220h160v-160H400v160ZM180-400h160v-160H180v160Zm440 0h160v-160H620v160ZM180-180h160v-160H180v160Zm440 0h160v-160H620v160Z\"/>\n",
              "  </svg>\n",
              "    </button>\n",
              "\n",
              "  <style>\n",
              "    .colab-df-container {\n",
              "      display:flex;\n",
              "      gap: 12px;\n",
              "    }\n",
              "\n",
              "    .colab-df-convert {\n",
              "      background-color: #E8F0FE;\n",
              "      border: none;\n",
              "      border-radius: 50%;\n",
              "      cursor: pointer;\n",
              "      display: none;\n",
              "      fill: #1967D2;\n",
              "      height: 32px;\n",
              "      padding: 0 0 0 0;\n",
              "      width: 32px;\n",
              "    }\n",
              "\n",
              "    .colab-df-convert:hover {\n",
              "      background-color: #E2EBFA;\n",
              "      box-shadow: 0px 1px 2px rgba(60, 64, 67, 0.3), 0px 1px 3px 1px rgba(60, 64, 67, 0.15);\n",
              "      fill: #174EA6;\n",
              "    }\n",
              "\n",
              "    .colab-df-buttons div {\n",
              "      margin-bottom: 4px;\n",
              "    }\n",
              "\n",
              "    [theme=dark] .colab-df-convert {\n",
              "      background-color: #3B4455;\n",
              "      fill: #D2E3FC;\n",
              "    }\n",
              "\n",
              "    [theme=dark] .colab-df-convert:hover {\n",
              "      background-color: #434B5C;\n",
              "      box-shadow: 0px 1px 3px 1px rgba(0, 0, 0, 0.15);\n",
              "      filter: drop-shadow(0px 1px 2px rgba(0, 0, 0, 0.3));\n",
              "      fill: #FFFFFF;\n",
              "    }\n",
              "  </style>\n",
              "\n",
              "    <script>\n",
              "      const buttonEl =\n",
              "        document.querySelector('#df-ea456e67-3306-47c0-92c4-3a41ff950044 button.colab-df-convert');\n",
              "      buttonEl.style.display =\n",
              "        google.colab.kernel.accessAllowed ? 'block' : 'none';\n",
              "\n",
              "      async function convertToInteractive(key) {\n",
              "        const element = document.querySelector('#df-ea456e67-3306-47c0-92c4-3a41ff950044');\n",
              "        const dataTable =\n",
              "          await google.colab.kernel.invokeFunction('convertToInteractive',\n",
              "                                                    [key], {});\n",
              "        if (!dataTable) return;\n",
              "\n",
              "        const docLinkHtml = 'Like what you see? Visit the ' +\n",
              "          '<a target=\"_blank\" href=https://colab.research.google.com/notebooks/data_table.ipynb>data table notebook</a>'\n",
              "          + ' to learn more about interactive tables.';\n",
              "        element.innerHTML = '';\n",
              "        dataTable['output_type'] = 'display_data';\n",
              "        await google.colab.output.renderOutput(dataTable, element);\n",
              "        const docLink = document.createElement('div');\n",
              "        docLink.innerHTML = docLinkHtml;\n",
              "        element.appendChild(docLink);\n",
              "      }\n",
              "    </script>\n",
              "  </div>\n",
              "\n",
              "\n",
              "  <div id=\"id_1b8f4a64-f897-4663-af77-c90d61441c17\">\n",
              "    <style>\n",
              "      .colab-df-generate {\n",
              "        background-color: #E8F0FE;\n",
              "        border: none;\n",
              "        border-radius: 50%;\n",
              "        cursor: pointer;\n",
              "        display: none;\n",
              "        fill: #1967D2;\n",
              "        height: 32px;\n",
              "        padding: 0 0 0 0;\n",
              "        width: 32px;\n",
              "      }\n",
              "\n",
              "      .colab-df-generate:hover {\n",
              "        background-color: #E2EBFA;\n",
              "        box-shadow: 0px 1px 2px rgba(60, 64, 67, 0.3), 0px 1px 3px 1px rgba(60, 64, 67, 0.15);\n",
              "        fill: #174EA6;\n",
              "      }\n",
              "\n",
              "      [theme=dark] .colab-df-generate {\n",
              "        background-color: #3B4455;\n",
              "        fill: #D2E3FC;\n",
              "      }\n",
              "\n",
              "      [theme=dark] .colab-df-generate:hover {\n",
              "        background-color: #434B5C;\n",
              "        box-shadow: 0px 1px 3px 1px rgba(0, 0, 0, 0.15);\n",
              "        filter: drop-shadow(0px 1px 2px rgba(0, 0, 0, 0.3));\n",
              "        fill: #FFFFFF;\n",
              "      }\n",
              "    </style>\n",
              "    <button class=\"colab-df-generate\" onclick=\"generateWithVariable('comparison')\"\n",
              "            title=\"Generate code using this dataframe.\"\n",
              "            style=\"display:none;\">\n",
              "\n",
              "  <svg xmlns=\"http://www.w3.org/2000/svg\" height=\"24px\"viewBox=\"0 0 24 24\"\n",
              "       width=\"24px\">\n",
              "    <path d=\"M7,19H8.4L18.45,9,17,7.55,7,17.6ZM5,21V16.75L18.45,3.32a2,2,0,0,1,2.83,0l1.4,1.43a1.91,1.91,0,0,1,.58,1.4,1.91,1.91,0,0,1-.58,1.4L9.25,21ZM18.45,9,17,7.55Zm-12,3A5.31,5.31,0,0,0,4.9,8.1,5.31,5.31,0,0,0,1,6.5,5.31,5.31,0,0,0,4.9,4.9,5.31,5.31,0,0,0,6.5,1,5.31,5.31,0,0,0,8.1,4.9,5.31,5.31,0,0,0,12,6.5,5.46,5.46,0,0,0,6.5,12Z\"/>\n",
              "  </svg>\n",
              "    </button>\n",
              "    <script>\n",
              "      (() => {\n",
              "      const buttonEl =\n",
              "        document.querySelector('#id_1b8f4a64-f897-4663-af77-c90d61441c17 button.colab-df-generate');\n",
              "      buttonEl.style.display =\n",
              "        google.colab.kernel.accessAllowed ? 'block' : 'none';\n",
              "\n",
              "      buttonEl.onclick = () => {\n",
              "        google.colab.notebook.generateWithVariable('comparison');\n",
              "      }\n",
              "      })();\n",
              "    </script>\n",
              "  </div>\n",
              "\n",
              "    </div>\n",
              "  </div>\n"
            ],
            "application/vnd.google.colaboratory.intrinsic+json": {
              "type": "dataframe",
              "variable_name": "comparison",
              "summary": "{\n  \"name\": \"comparison\",\n  \"rows\": 5,\n  \"fields\": [\n    {\n      \"column\": \"Strategy\",\n      \"properties\": {\n        \"dtype\": \"string\",\n        \"num_unique_values\": 5,\n        \"samples\": [\n          \"CharacterTextSplitter\",\n          \"Token-based\",\n          \"RecursiveCharacterTextSplitter\"\n        ],\n        \"semantic_type\": \"\",\n        \"description\": \"\"\n      }\n    },\n    {\n      \"column\": \"Number of Chunks\",\n      \"properties\": {\n        \"dtype\": \"number\",\n        \"std\": 0,\n        \"min\": 7,\n        \"max\": 9,\n        \"num_unique_values\": 3,\n        \"samples\": [\n          8,\n          7,\n          9\n        ],\n        \"semantic_type\": \"\",\n        \"description\": \"\"\n      }\n    },\n    {\n      \"column\": \"Avg Chunk Length (chars)\",\n      \"properties\": {\n        \"dtype\": \"number\",\n        \"std\": 38.71818694102294,\n        \"min\": 196.8,\n        \"max\": 305.0,\n        \"num_unique_values\": 5,\n        \"samples\": [\n          253.3,\n          305.0,\n          196.8\n        ],\n        \"semantic_type\": \"\",\n        \"description\": \"\"\n      }\n    }\n  ]\n}"
            }
          },
          "metadata": {},
          "execution_count": 11
        }
      ],
      "source": [
        "import pandas as pd\n",
        "\n",
        "comparison = pd.DataFrame({\n",
        "    \"Strategy\": [\n",
        "        \"Fixed-size (scratch)\",\n",
        "        \"CharacterTextSplitter\",\n",
        "        \"RecursiveCharacterTextSplitter\",\n",
        "        \"Sentence-based\",\n",
        "        \"Token-based\"\n",
        "    ],\n",
        "    \"Number of Chunks\": [\n",
        "        len(fixed_chunks),\n",
        "        len(char_chunks),\n",
        "        len(recursive_chunks),\n",
        "        len(sentence_chunks),\n",
        "        len(token_chunks)\n",
        "    ],\n",
        "    \"Avg Chunk Length (chars)\": [\n",
        "        round(sum(len(c) for c in fixed_chunks) / len(fixed_chunks), 1),\n",
        "        round(sum(len(c) for c in char_chunks) / len(char_chunks), 1),\n",
        "        round(sum(len(c) for c in recursive_chunks) / len(recursive_chunks), 1),\n",
        "        round(sum(len(c) for c in sentence_chunks) / len(sentence_chunks), 1),\n",
        "        round(sum(len(c) for c in token_chunks) / len(token_chunks), 1)\n",
        "    ]\n",
        "})\n",
        "\n",
        "comparison"
      ],
      "id": "oJ1ZT7r-tu7R"
    },
    {
      "cell_type": "code",
      "execution_count": 12,
      "metadata": {
        "colab": {
          "base_uri": "https://localhost:8080/",
          "height": 526
        },
        "id": "6UOtY3MTtu7S",
        "outputId": "eee76603-6896-4d62-8d34-5a183e393b00"
      },
      "outputs": [
        {
          "output_type": "display_data",
          "data": {
            "text/plain": [
              "<Figure size 800x500 with 1 Axes>"
            ],
            "image/png": "...\n"
          },
          "metadata": {}
        }
      ],
      "source": [
        "import matplotlib.pyplot as plt\n",
        "\n",
        "plt.figure(figsize=(8, 5))\n",
        "plt.bar(comparison[\"Strategy\"], comparison[\"Number of Chunks\"])\n",
        "plt.xticks(rotation=30, ha=\"right\")\n",
        "plt.ylabel(\"Number of Chunks\")\n",
        "plt.title(\"Number of Chunks Produced by Each Strategy\")\n",
        "plt.tight_layout()\n",
        "plt.show()"
      ],
      "id": "6UOtY3MTtu7S"
    }
  ],
  "metadata": {
    "kernelspec": {
      "display_name": "Python 3",
      "language": "python",
      "name": "python3"
    },
    "language_info": {
      "name": "python",
      "version": "3.10"
    },
    "colab": {
      "provenance": []
    }
  },
  "nbformat": 4,
  "nbformat_minor": 5
}

{'cells': [{'cell_type': 'code',
   'execution_count': 1,
   'metadata': {'colab': {'base_uri': 'https://localhost:8080/'},
    'id': '2qJm5sdEtu6z',
    'outputId': '8d37cebc-cc98-40e9-8af2-642739771e14'},
   'outputs': [{'output_type': 'stream',
     'name': 'stdout',
     'text': ['\x1b[2K     \x1b━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\x1b \x1b88.8/88.8 kB\x1b \x1b1.9 MB/s\x1b eta \x1b0:00:00\x1b\n',
      '\x1b[2K   \x1b━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\x1b \x1b395.5/395.5 kB\x1b \x1b9.4 MB/s\x1b eta \x1b0:00:00\x1b\n',
      '\x1b[2K   \x1b━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\x1b \x1b341.3/341.3 kB\x1b \x1b9.5 MB/s\x1b eta \x1b0:00:00\x1b\n',
      '\x1b[?25h']}],
   'source': ['!pip install -q langchain langchain-text-splitters tiktoken pypdf fpdf2'],
   'id': '2qJm5sdEtu6z'},
  {'cell_type': 'code',
   'execution_count': 2,
   'metadata': {'colab': {'base_uri': 'https://localhost:8080/'},
    'id': 'CAfbB3bYtu67',
    'outputId': '138afab7-b79c-4683-c423-628b9360b01